# Polynomial and spline regression

Simulated example adapted from https://tgstewart.cloud/ipm/nonlinear.html, then and example with the bike rental data.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, PolynomialFeatures, SplineTransformer

RANDOM_STATE = 2345

## Simulated data: $y = \sin(2\pi x) + \epsilon$

In this example, we are simulating the curve $y = \sin(2\pi x)$ with deliberately added noise ($\epsilon$). 

In [ ]:
rng = np.random.default_rng(7)
n = 100
x = rng.uniform(size=n) # Draw 100 samples between 0 and 1 from a uniform distribution 
y = np.sin(2 * np.pi * x) + rng.normal(0, 0.1, n) # Compute y = sin(2\pi x) + random number drawn from a Gaussian with 0 mean and 0.1 std dev
X = x[:, None]

grid = np.linspace(0, 1, 300)[:, None]

# Here we are writing a function to generate a plot of this function
# We will use it multiple times 
def plot_fit(yhat=None, title="", xs=grid, ax=None):
    ax = ax or plt.subplots()[1]
    ax.scatter(x, y, color="steelblue", alpha=0.6)
    ax.plot(xs, np.sin(2 * np.pi * xs), color="red", ls="--", label="truth")
    if yhat is not None:
        ax.plot(xs, yhat, color="darkorange", lw=3, label="fit")
    ax.set(xlabel="x", ylabel="y", title=title, ylim=(-1.6, 1.6))
    ax.legend(frameon=False)

plot_fit(title="Simulated data")

## Polynomials

Now, we are going to try and fit this curve using a polynomial regression. We will try FOUR different degrees and compare them:
* d=1 (Linear)
* d=2 (Quadratic - x^2)
* d=3 (Cubic - x^3)
* d=20 (x^20 is the highest order polynomial)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for ax, d in zip(axes.flat, [1, 2, 3, 20]):
    poly = make_pipeline(PolynomialFeatures(d, include_bias=True), LinearRegression()).fit(X, y)
    plot_fit(poly.predict(grid), f"Polynomial, degree {d}", ax=ax) # Note: This is the function we defined earlier
fig.tight_layout()

In [ ]:
# Just beyond the data: the degree-20 fit falls apart, degree 3 does not

# Here, we are making the grid that we pass into the plotting function just a little bit wider
wide = np.linspace(-0.05, 1.05, 400)[:, None]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, d in zip(axes, [3, 20]):
    poly = make_pipeline(PolynomialFeatures(d, include_bias=False), LinearRegression()).fit(X, y)
    plot_fit(poly.predict(wide), f"Polynomial, degree {d}", xs=wide, ax=ax)
    ax.axvspan(x.min(), x.max(), color="gray", alpha=0.1)
    ax.set(ylim=(-5, 2))
fig.tight_layout()

## Linear spline

Now, let's try fitting this using a linear spline with the following parameters:

* Basis: $x,\ (x - c_1)_+,\ (x - c_2)_+$
* c1 at the 1/3 quantile and c2 at the 2/3 quantile

In [ ]:
c1, c2 = np.quantile(x, [0.33, 0.67])

# Function Transformer creates a transformer from an arbitrary callable
# Here, it uses lambda np.hstack to return an array of three values for each x 
# This constructs the basis functions for the fit step
linear_basis = FunctionTransformer(lambda x: np.hstack([x, np.maximum(x - c1, 0), np.maximum(x - c2, 0)]))
linear_spline = make_pipeline(linear_basis, LinearRegression()).fit(X, y)
plot_fit(linear_spline.predict(grid), "Linear spline, knots at the .33 and .67 quantiles")

In [ ]:
# We can inspect the results of the FunctionTransformer to see the basis functions if we want
test_x = 0.50
print(c1)
np.hstack([test_x, np.maximum(test_x - c1, 0), np.maximum(test_x - c2, 0)])

In [ ]:
# slope is b1 before c1, b1 + b2 between the knots, b1 + b2 + b3 after c2
# We can manually check this with the output of the pipeline function 
b = linear_spline[-1].coef_
b.round(2), b.cumsum().round(2)

In [ ]:
# What if we put the knots at different quantiles

c1, c2 = np.quantile(x, [0.25, 0.75])

linear_basis = FunctionTransformer(lambda x: np.hstack([x, np.maximum(x - c1, 0), np.maximum(x - c2, 0)]))
linear_spline = make_pipeline(linear_basis, LinearRegression()).fit(X, y)
plot_fit(linear_spline.predict(grid), "Linear spline, knots at the .25 and .75 quantiles")

In [ ]:
# Looking again at the slopes 
b = linear_spline[-1].coef_
b.round(2), b.cumsum().round(2)

## Cubic spline
Basis: $x,\ x^2,\ x^3,\ (x - c_1)_+^3,\ (x - c_2)_+^3$

In [ ]:
# Reset c1 and c2 to the original quantile functions and recalculate the linear spline for use later
c1, c2 = np.quantile(x, [0.33, 0.67])
linear_basis = FunctionTransformer(lambda x: np.hstack([x, np.maximum(x - c1, 0), np.maximum(x - c2, 0)]))
linear_spline = make_pipeline(linear_basis, LinearRegression()).fit(X, y)

# As we did before, we can construct the basis functions for the cubic spline
cubic_basis = FunctionTransformer(
    lambda x: np.hstack([x, x**2, x**3, np.maximum(x - c1, 0) ** 3, np.maximum(x - c2, 0) ** 3]))
cubic_spline = make_pipeline(cubic_basis, LinearRegression()).fit(X, y)
plot_fit(cubic_spline.predict(grid), "Cubic spline, same two knots")

## Natural cubic spline
scikit-learn has no natural spline, so this uses `cr()` in a statsmodels formula (as in Week 5).

In [ ]:
natural_spline = smf.ols("y ~ cr(x, knots=[c1, c2])", data={"x": x, "y": y}).fit()
plot_fit(natural_spline.predict({"x": grid.ravel()}), "Natural cubic spline, same two knots")

In [ ]:
# Beyond the data: the cubic spline keeps curving, the natural spline is linear
wide = np.linspace(-0.5, 1.5, 400)[:, None]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
plot_fit(cubic_spline.predict(wide), "Cubic spline", xs=wide, ax=axes[0])
plot_fit(natural_spline.predict({"x": wide.ravel()}), "Natural cubic spline", xs=wide, ax=axes[1])
for ax in axes:
    ax.set(ylim=(-6, 6))

## B-splines
`SplineTransformer` uses B-spline basis. Any piecewise polynomial with continuous first and second derivatives can be 
written as a linear combination of of B-splines. 

We'll see below that we get the same predictions. The coefficients will be different, but the predictions are the same. Recall what we learned last week about different linear transformations. 

In [ ]:
#Note: Using SplineTransformers is preferred. 

knots = np.array([x.min(), c1, c2, x.max()])[:, None]

linear_bspline = make_pipeline(SplineTransformer(degree=1, knots=knots), LinearRegression()).fit(X, y)
cubic_bspline = make_pipeline(SplineTransformer(degree=3, knots=knots), LinearRegression()).fit(X, y)

print("linear spline, same predictions:", np.allclose(linear_spline.predict(X), linear_bspline.predict(X)))
print("cubic spline,  same predictions:", np.allclose(cubic_spline.predict(X), cubic_bspline.predict(X)))

## Bike rentals dataset

Now let's apply some of this to the bike rentals dataset

In [ ]:
bike = pd.read_csv("../data/bike-sharing-daily.csv")

def rmse(model, X, y):
    return float(mean_squared_error(y, model.predict(X)) ** 0.5)

### Polynomial regression

In [ ]:
# We are going to predict rental count as a function of temperature.

x1 = bike[["temp"]].values
yy = bike["cnt"].values
grid = np.linspace(x1.min(), x1.max(), 200)[:, None]

x1_tr, x1_te, y1_tr, y1_te = train_test_split(x1, yy, test_size=0.25, random_state=RANDOM_STATE)

plt.figure(figsize=(9, 5))
plt.scatter(x1, yy, s=8, alpha=0.3)
for d in [1, 2, 3, 10]:
    m = make_pipeline(PolynomialFeatures(d), LinearRegression()).fit(x1_tr, y1_tr)
    plt.plot(grid, m.predict(grid), lw=2,
             label=f"degree {d} (test RMSE {rmse(m, x1_te, y1_te):.0f})")
plt.xlabel("normalized temperature"); plt.ylabel("daily rentals"); plt.legend()
plt.tight_layout()

### Splines

In [ ]:
# Cubic splines with sklearn's B-spline basis; n_knots includes the two boundary knots.

plt.figure(figsize=(9, 5))
plt.scatter(x1, yy, s=8, alpha=0.25)
for k in [3, 5, 10]:
    # Here we aren't specifying where the knots are, just the number and how to select them. 
    m = make_pipeline(SplineTransformer(degree=3, n_knots=k, knots="quantile"), LinearRegression()).fit(x1_tr, y1_tr)
    plt.plot(grid, m.predict(grid), lw=2,
             label=f"cubic spline, {k} knots (test RMSE {rmse(m, x1_te, y1_te):.0f})")
plt.xlabel("normalized temperature"); plt.ylabel("daily rentals"); plt.legend()
plt.tight_layout()

### Which model should we choose?

In [ ]:
from sklearn.model_selection import KFold, cross_val_score, validation_curve

# The test RMSEs above come from one split and move with RANDOM_STATE, so compare with 10-fold CV.
cv = KFold(10, shuffle=True, random_state=RANDOM_STATE)
scores, n_params = {}, {}

for d in [1, 2, 3, 10]:
    name = f"polynomial, degree {d}"
    model = make_pipeline(PolynomialFeatures(d), LinearRegression())
    scores[name] = -cross_val_score(model, x1, yy, cv=cv, scoring="neg_root_mean_squared_error")
    n_params[name] = d + 1

for k in [3, 5, 10]:
    name = f"cubic spline, {k} knots"
    model = make_pipeline(SplineTransformer(degree=3, n_knots=k, knots="quantile"), LinearRegression())
    scores[name] = -cross_val_score(model, x1, yy, cv=cv, scoring="neg_root_mean_squared_error")
    n_params[name] = k + 2

cv_table = pd.DataFrame({"parameters": n_params,
                         "CV RMSE": {m: np.mean(s) for m, s in scores.items()},
                         "SE": {m: np.std(s, ddof=1) / np.sqrt(len(s)) for m, s in scores.items()}})
cv_table.round(2)

Only the straight line is clearly worse. Every curved model's RMSE is within one standard error of the best model's RMSE.
With no real difference in error, choose the simplest: the quadratic with three parameters. 